# 07 - Deploy a Demo App with Gradio

**Learning objective:** Wrap the ReAct agent in a minimal web UI and demo it.

We will:
1. Define a Gradio interface around the ReAct loop
2. Launch it on localhost:7860
3. Demo to the room

To access from outside the Jupyter instance, use SSH port forwarding:
```bash
ssh -L 7860:localhost:7860 root@<public-ip>
```
Then open http://localhost:7860 in your browser.

In [ ]:
import os
import sys
import subprocess
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

# Database connection
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)

# Embeddings client
mi_dir = "../iac_snippets/managed_inference"
result = subprocess.run(["tofu", "output", "-raw", "endpoint_url"], cwd=mi_dir, capture_output=True, text=True)
endpoint_url = result.stdout.strip()

from src.embeddings import EmbeddingsClient

embedding_client = EmbeddingsClient(
    client=OpenAI(base_url=endpoint_url, api_key=os.environ["SCW_SECRET_KEY"]),
    model="baai/bge-multilingual-gemma2:fp16",
)

llm_client = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

from src.tools import ToolKit

toolkit = ToolKit(conn=conn, embeddings_client=embedding_client, llm_client=llm_client)

print("Components ready.")

In [ ]:
import gradio as gr
from src.react_loop import run_react_loop


def analyze_medications(medications: str, population: str) -> str:
    """Run the ReAct agent and format the output."""
    query = f"Patient medications: {medications}"
    if population and population != "None":
        query += f". Patient population attributes: {population}"
    query += ". List all drug interactions and population-specific warnings with severity and FDA label citations."

    result = run_react_loop(
        query=query,
        toolkit=toolkit,
        llm_client=llm_client,
    )

    # Format output
    output_parts = []

    # Trace
    output_parts.append("## Agent Trace\n")
    for i, entry in enumerate(result["trace"]):
        output_parts.append(f"**Step {i + 1}:** {entry['act'][:100]}")

    output_parts.append("\n## Final Report\n")
    output_parts.append(result["final_response"])

    return "\n".join(output_parts)


demo = gr.Interface(
    fn=analyze_medications,
    inputs=[
        gr.Textbox(
            label="Medications",
            placeholder="e.g., warfarin, aspirin, ibuprofen",
            value="warfarin, aspirin, ibuprofen",
        ),
        gr.Dropdown(
            label="Population Attributes",
            choices=[
                "None",
                "pregnancy (32 weeks)",
                "pediatric",
                "geriatric",
                "renal impairment",
                "hepatic impairment",
            ],
            value="pregnancy (32 weeks)",
        ),
    ],
    outputs=gr.Markdown(label="Analysis"),
    title="Drug Interaction & Population Warning Agent",
    description=(
        "ReAct agent backed by FDA drug labels (openFDA SPL), pgvector, and Mistral Small 3.2. "
        "Every finding cites a specific FDA label section."
    ),
)

demo.launch(server_name="0.0.0.0", server_port=7860, share=False)
print("Gradio app running on http://localhost:7860")
print("For external access: ssh -L 7860:localhost:7860 root@<public-ip>")

## You should now have

- [x] A running Gradio app wrapping the ReAct agent
- [x] Interactive medication input + population attributes
- [x] Severity-first report with FDA label citations

**Next:** `99_teardown.ipynb` -- clean up all resources